In [13]:
import pandas as pd
import pickle

from sklearn.impute import SimpleImputer
import joblib

import folium

In [4]:
with open("lfbo/tracks_clustered.pkl", "rb") as f:
    valid_tracks = pickle.load(f)

In [5]:
from src.features import compute_features

feature_rows = []
for track in valid_tracks:
    row = compute_features(track, track.cluster_id)
    if row is not None:
        feature_rows.append(row)

df_features = pd.DataFrame(feature_rows)

print(f"Feature matrix: {df_features.shape}")
print(f"\nFeature columns:")
print([c for c in df_features.columns if c not in ["icao24", "callsign", "source_date"]])

print(f"\nMissing values:")
print(df_features.isnull().sum()[df_features.isnull().sum() > 0])

Feature matrix: (1546, 28)

Feature columns:
['track_length_km', 'straight_line_km', 'sinuosity', 'min_dist_to_lfbo_km', 'lateral_deviation_deg', 'alt_mean_m', 'alt_max_m', 'alt_min_m', 'alt_range_m', 'alt_std_m', 'alt_net_change_m', 'alt_slope', 'speed_mean_ms', 'speed_std_ms', 'speed_min_ms', 'speed_max_ms', 'speed_cv', 'heading_change_rate', 'duration_min', 'point_count', 'cluster_id', 'is_noise', 'is_arriving', 'is_departing', 'is_transiting']

Missing values:
sinuosity                87
lateral_deviation_deg    80
alt_mean_m               20
alt_max_m                20
alt_min_m                20
alt_range_m              20
alt_std_m                20
alt_net_change_m         20
alt_slope                20
speed_mean_ms            50
speed_std_ms             51
speed_min_ms             50
speed_max_ms             50
speed_cv                 51
heading_change_rate      51
dtype: int64


In [10]:
meta_cols = ["icao24", "callsign", "source_date", "cluster_id", "is_noise", "is_arriving", "is_departing", "is_transiting"]

feature_cols = [c for c in df_features.columns 
                if c not in meta_cols]

print(f"Feature columns for model: {len(feature_cols)}")
print(feature_cols)

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(df_features[feature_cols])
df_imputed = pd.DataFrame(X_imputed, columns=feature_cols)

print("\nKey feature distributions:")
print(df_imputed[["sinuosity", "alt_range_m", "speed_cv", "heading_change_rate", "min_dist_to_lfbo_km"]].describe().round(2))

joblib.dump(imputer, "lfbo/processed/imputer.pkl")
df_features.to_parquet("lfbo/processed/features.parquet", index=False)
print("\nSaved features.parquet")

Feature columns for model: 20
['track_length_km', 'straight_line_km', 'sinuosity', 'min_dist_to_lfbo_km', 'lateral_deviation_deg', 'alt_mean_m', 'alt_max_m', 'alt_min_m', 'alt_range_m', 'alt_std_m', 'alt_net_change_m', 'alt_slope', 'speed_mean_ms', 'speed_std_ms', 'speed_min_ms', 'speed_max_ms', 'speed_cv', 'heading_change_rate', 'duration_min', 'point_count']

Key feature distributions:
       sinuosity  alt_range_m  speed_cv  heading_change_rate  \
count    1546.00      1546.00   1546.00              1546.00   
mean        1.28      1290.45      0.05                 0.53   
std         7.21      2243.72      0.10                 1.52   
min         1.00         0.00      0.00                 0.00   
25%         1.00         3.05      0.00                 0.05   
50%         1.00        18.29      0.01                 0.11   
75%         1.01      1534.67      0.02                 0.38   
max       282.95     10050.78      0.54                31.98   

       min_dist_to_lfbo_km  
cou

### Scaling the features

In [11]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
X_final = robust_scaler.fit_transform(df_imputed[feature_cols])
df_final = pd.DataFrame(X_final, columns=feature_cols)

print("Feature ranges after RobustScaler:")
print(df_final.describe().loc[["min","50%","max"]].round(2))

joblib.dump(robust_scaler, "lfbo/processed/robust_scaler.pkl")
print("\nSaved robust_scaler.pkl")

Feature ranges after RobustScaler:
     track_length_km  straight_line_km  sinuosity  min_dist_to_lfbo_km  \
min            -1.14             -1.04      -0.04                -0.84   
50%             0.00              0.00       0.00                 0.00   
max             5.71              0.86   41459.11                 2.43   

     lateral_deviation_deg  alt_mean_m  alt_max_m  alt_min_m  alt_range_m  \
min                  -0.19       -5.00      -6.93      -3.04        -0.01   
50%                   0.00        0.00       0.00       0.00         0.00   
max                  16.51        8.82      11.82       5.40         6.55   

     alt_std_m  alt_net_change_m  alt_slope  speed_mean_ms  speed_std_ms  \
min      -0.01            -716.8    -670.93          -7.46         -0.52   
50%       0.00               0.0       0.00           0.00          0.00   
max       6.47             791.4     750.47           1.43         22.24   

     speed_min_ms  speed_max_ms  speed_cv  heading_cha

In [ ]:
max_sin_idx = df_features["sinuosity"].idxmax()
max_sin_track = valid_tracks[max_sin_idx]
print(f"\nMax sinuosity track:")
print(f"  icao24:    {max_sin_track.icao24}")
print(f"  callsign:  {max_sin_track.callsign}")
print(f"  sinuosity: {df_features['sinuosity'].max():.1f}")
print(f"  duration:  {max_sin_track.duration_seconds/60:.1f} min")


Max sinuosity track:
  icao24:    3818fa
  callsign:  FWWEY   
  sinuosity: 282.9
  duration:  97.7 min


## Anomaly Preview — Airbus Flight Test Traffic

Before running any model, manual inspection of the sinuosity feature already surfaces a compelling anomaly candidate:

icao24: 3818fa | callsign: FWWEY | sinuosity: 282.9 | duration: 97.7 min

The FW prefix identifies a French experimental aircraft registration. FWWEY is an Airbus flight test aircraft operating from LFBO. A sinuosity of 282 indicates the aircraft flew a total distance 282x greater than its straight-line displacement which is consistent with a structured flight test maneuver profile.

This demonstrates that simple univariate feature inspection already surfaces operationally meaningful anomalies before any Machine Learning model is applied.

In [14]:
LFBO_LAT, LFBO_LON = 43.6293, 1.3673

m = folium.Map(location=[LFBO_LAT, LFBO_LON], zoom_start=9)

folium.Marker([LFBO_LAT, LFBO_LON], popup="LFBO Toulouse-Blagnac", icon=folium.Icon(color="red", icon="plane")).add_to(m)

pts = max_sin_track.points.dropna(
    subset=["lat", "lon"]
)[["lat", "lon"]].values.tolist()

folium.PolyLine(pts, color="red", weight=2, opacity=0.8, 
                popup=f"FWWEY | Airbus Flight Test<br>"
                f"Sinuosity: 282.9<br>"
                f"Duration: 97.7 min"
                ).add_to(m)

m.save("figures/anomaly_fwwey.html")
print("Saved anomaly_fwwey.html")

Saved anomaly_fwwey.html
